# Comparacion de modelos de clasificacion HistGradientBoosting, LightGBM, CatBoost, XGBoost

In [1]:
# Standard libraries
import os
import warnings

In [2]:

# Data manipulation and numerical computation
import numpy as np
import pandas as pd


In [3]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, confusion_matrix, classification_report
)
from sklearn.calibration import CalibratedClassifierCV

In [5]:

from typing import Dict, Any, List, Optional, Tuple
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier

# --- add near imports ---
import os, json, re
from pathlib import Path
from datetime import datetime
import joblib

import os, json
from pathlib import Path
from datetime import datetime
import joblib
import re



In [6]:
# Statistical distributions
from scipy.stats import loguniform, randint, uniform

# Typing utilities
from typing import Dict, Any, List, Optional, Tuple

In [7]:
sns.set(style="ticks", context="notebook", palette="deep")
pd.set_option('display.max_columns', None)
palette = {'Bad':'#b2182b','Poor':'#d6604d','Moderate':'#f1a340','Good':'#5aae61','High':'#1b7837'}

In [8]:
path = "../../../data/processed/"
dfs = {}

# read 03_CLEAN_COMPLETE_DF.parquet
df1 = pd.read_parquet(os.path.join(path, "taxones_pressure_epm_train.parquet"))
df2 = pd.read_parquet(os.path.join(path, "taxones_pressure_epm_predict.parquet"))
df = pd.concat([df1, df2], ignore_index=True)

In [9]:
# change unassessed values to NaN
df = df.replace("Unassessed", np.nan)
df = df.replace("None", np.nan)

# index SamplingOperations_code
df = df.set_index('SamplingOperations_code')
 
# DROP HERlvl2Code	Altitude Longitude_Lambert93	Latitude_Lambert93	Watershed	CodeDepartement	HERlvl1Code
df = df.drop(columns=['HERlvl2Code', 'HERlvl2Name', 'HERlvl1Name', 'Altitude','Longitude_Lambert93','Latitude_Lambert93','Watershed','CodeDepartement', 'Date_SamplingOperation'])
df

CodeSite_SamplingOperations_x  \
SamplingOperations_code                                 
S02000008_20170703                          S02000008   
S02000008_20200708                          S02000008   
S02000010_20070906                          S02000010   
S02000010_20090721                          S02000010   
S02000010_20110723                          S02000010   
...                                               ...   
S06940940_20100708                          S06940940   
S06940940_20230623                          S06940940   
S06960950_20160629                          S06960950   
S06960950_20180719                          S06960950   
S06970900_20150601                          S06970900   

                         TotalAbundance_SamplingOperation  Achaa01  Achac01  \
SamplingOperations_code                                                       
S02000008_20170703                                    405      NaN      NaN   
S02000008_20200708                                    400      NaN      NaN   
S02000010_20070906                                    400      NaN      NaN   
S02000010_20090721                                    400      NaN      NaN   
S02000010_20110723                                    398      NaN      NaN   
...                                                   ...      ...      ...   
S06940940_20100708                                    438      NaN      NaN   
S06940940_20230623                                    408      NaN      NaN   
S06960950_20160629                                    401      NaN      NaN   
S06960950_20180719                                    416      NaN      NaN   
S06970900_20150601                                    432      NaN      NaN   

                         Achaf01    Achaf02  Achal01  Acham01  Achan01  \
SamplingOperations_code                                                  
S02000008_20170703           NaN        NaN      NaN      NaN      NaN   
S02000008_20200708           NaN        NaN      NaN      NaN      NaN   
S02000010_20070906           NaN        NaN      NaN      NaN      NaN   
S02000010_20090721           NaN        NaN      NaN      NaN      NaN   
S02000010_20110723           NaN        NaN      NaN      NaN      NaN   
...                          ...        ...      ...      ...      ...   
S06940940_20100708           NaN        NaN      NaN      NaN      NaN   
S06940940_20230623           NaN        NaN      NaN      NaN      NaN   
S06960950_20160629           NaN        NaN      NaN      NaN      NaN   
S06960950_20180719           NaN  19.230769      NaN      NaN      NaN   
S06970900_20150601           NaN        NaN      NaN      NaN      NaN   

                         Achat01   Achat02  Achat03  Achba01  Achbi01  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S02000010_20090721           NaN       NaN      NaN      NaN      NaN   
S02000010_20110723           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN       NaN      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN  6.944444      NaN      NaN      NaN   

                         Achbi02   Achbr01  Achca01  Achca02  Achca03  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      N

Short answer: drop IBD* when fitting any imputer. Using the target to fill predictors creates leakage and over-optimistic CV; at inference the target is unknown so the imputer would use information you won’t have.

When you still want “best” fills for prediction, do this:

# For each region

In [10]:
import cleandf

In [11]:
# FastSISReducer: ultra-fast pre-imputation feature screener
# pip install numpy pandas scikit-learn
from __future__ import annotations
import re, numpy as np, pandas as pd
from dataclasses import dataclass
from typing import List, Optional, Tuple
from sklearn.base import BaseEstimator, TransformerMixin

# ---- column inference (same rules you used) ----
def _infer_columns(df: pd.DataFrame) -> Tuple[List[str], List[str], List[str], List[str], Optional[str]]:
    cols = df.columns.tolist()
    ibd_cols = [c for c in ["IBD","IBD_EQR","IBD_EQR_Status"] if c in cols]
    effort_col = "TotalAbundance_SamplingOperation" if "TotalAbundance_SamplingOperation" in cols else None
    status_cols = [c for c in cols if "Status" in c]
    chem_cols = [c for c in cols if c.startswith(("Mean90Days_","Mean180Days_","Mean1Y_")) or c.endswith("_XOMP")]
    group_cols = [c for c in ["HERlvl1Name","HERlvl1Code","CodeSite_SamplingOperations_x","CodeSite_SamplingOperations_y"] if c in cols]
    reserved = set(ibd_cols + status_cols + chem_cols + group_cols + ([effort_col] if effort_col else []))
    taxa_cols = [c for c in cols if c not in reserved and c and c[0].isalpha() and any(ch.isdigit() for ch in c)]
    if "Streamsize" in cols and "Streamsize" not in chem_cols:
        chem_cols.append("Streamsize")
    return taxa_cols, status_cols, chem_cols, group_cols, effort_col

# ---- fast helpers ----
def _spearman_on_pairs(x: pd.Series, y: pd.Series) -> float:
    # pairwise complete obs
    m = x.notna() & y.notna()
    if m.sum() < 25: 
        return 0.0
    xr = x[m].rank(method="average")
    yr = y[m].rank(method="average")
    return xr.corr(yr)

def _status_to_ord(s: pd.Series) -> pd.Series:
    return s.astype("object").map({"Bad":1,"Poor":2,"Moderate":3,"Good":4,"High":5}).fillna(0).astype(np.int16)

@dataclass
class SISConfig:
    # Hard filters
    max_missing: float = 0.98       # drop if >98% NaN
    min_n_pairs: int = 50           # need at least 50 (x,y) pairs
    min_presence_frac: float = 0.005 # drop taxa with <0.5% positives after struct zeros
    # Pooling
    pool_prefix_len: int = 3
    min_pool_size: int = 4
    min_pool_prev: float = 0.02
    # Ranking
    subsample_rows: int = 20000     # speed cap; set 0 for all rows
    top_k: int = 300                # final kept features
    corr_drop: float = 0.98         # drop redundant by |rho|>0.98 w.r.t. earlier-kept feature
    random_state: int = 42

class FastSISReducer(BaseEstimator, TransformerMixin):
    def __init__(self, cfg: SISConfig = SISConfig()):
        self.cfg = cfg

    def fit(self, X: pd.DataFrame, y: pd.Series):
        taxa_cols, status_cols, chem_cols, group_cols, effort_col = _infer_columns(X)
        self.group_cols_ = group_cols
        self.effort_col_ = effort_col

        # optional subsample for speed
        if self.cfg.subsample_rows and len(X) > self.cfg.subsample_rows:
            rng = np.random.RandomState(self.cfg.random_state)
            idx = rng.choice(len(X), size=self.cfg.subsample_rows, replace=False)
            Xs, ys = X.iloc[idx], y.iloc[idx]
        else:
            Xs, ys = X, y

        # structural zeros only to compute presence
        if effort_col and effort_col in Xs.columns:
            mask_zero_eff = pd.to_numeric(Xs[effort_col], errors="coerce").fillna(0) <= 0
        else:
            mask_zero_eff = pd.Series(False, index=Xs.index)

        X0 = Xs.copy()
        if mask_zero_eff.any():
            X0.loc[mask_zero_eff, taxa_cols] = 0.0

        # 1) Hard screens
        keep_taxa = []
        for c in taxa_cols:
            s = pd.to_numeric(X0[c], errors="coerce")
            if s.isna().mean() > self.cfg.max_missing:
                continue
            pres = (s.fillna(0) > 0).mean()
            if pres < self.cfg.min_presence_frac:
                continue
            keep_taxa.append(c)

        # Pool ultra-rare taxa by prefix to salvage signal cheaply
        rare_buckets = {}
        for c in set(taxa_cols) - set(keep_taxa):
            pfx = re.sub(r'[^A-Za-z].*$', '', c)[:self.cfg.pool_prefix_len] or "RARE"
            rare_buckets.setdefault(pfx, []).append(c)

        pool_cols = []
        for pfx, cols in rare_buckets.items():
            if len(cols) < self.cfg.min_pool_size:
                continue
            colname = f"POOL_{pfx}"
            X0[colname] = pd.to_numeric(X0[cols], errors="coerce").fillna(0).sum(axis=1)
            if (X0[colname] > 0).mean() >= self.cfg.min_pool_prev:
                pool_cols.append(colname)

        # Chemistry + statuses basic screen
        chem_keep = [c for c in chem_cols if X0[c].isna().mean() <= self.cfg.max_missing]
        # 2) Rank by fast Spearman with IBD on observed pairs
        cand = keep_taxa + pool_cols + chem_keep + status_cols
        ranks = []
        for c in cand:
            if c in status_cols:
                s = _status_to_ord(X0[c])
            else:
                s = pd.to_numeric(X0[c], errors="coerce")
            n_pairs = (s.notna() & ys.notna()).sum()
            if n_pairs < self.cfg.min_n_pairs:
                continue
            r = abs(_spearman_on_pairs(s, ys))
            # weight by sqrt(n_pairs/N) to prefer stable estimates
            w = np.sqrt(n_pairs / len(X0))
            ranks.append((c, r * w, r, n_pairs))
        if not ranks:
            # fallback: keep small safe core
            self.selected_ = chem_keep[: min(50, len(chem_keep))]
            self.pools_ = {pc: [] for pc in pool_cols}
            return self

        R = pd.DataFrame(ranks, columns=["col","score","rho","n"]).sort_values("score", ascending=False)

        # 3) Redundancy pruning (very cheap): greedy |rho|>corr_drop vs already kept
        selected = []
        corr_cache = {}
        for col in R["col"]:
            if len(selected) >= self.cfg.top_k:
                break
            s_col = X0[col] if col not in status_cols else _status_to_ord(X0[col])
            s_col = pd.to_numeric(s_col, errors="coerce")
            redundant = False
            for kept in selected:
                key = (kept, col)
                if key in corr_cache:
                    rkk = corr_cache[key]
                else:
                    a = pd.to_numeric(X0[kept] if kept not in status_cols else _status_to_ord(X0[kept]), errors="coerce")
                    m = a.notna() & s_col.notna()
                    if m.sum() < 25:
                        rkk = 0.0
                    else:
                        rkk = abs(a[m].rank().corr(s_col[m].rank()))
                    corr_cache[key] = rkk
                if rkk >= self.cfg.corr_drop:
                    redundant = True
                    break
            if not redundant:
                selected.append(col)

        self.selected_ = selected
        self.status_cols_ = status_cols
        self.pools_ = {pc: rare_buckets.get(pc.replace("POOL_",""), []) for pc in pool_cols}
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        # materialize only what we kept + passthrough columns required later
        Z = X.copy()
        # structural zeros for pool building consistency
        taxa_cols, _, _, _, effort_col = _infer_columns(Z)
        if effort_col and effort_col in Z.columns:
            mask_zero_eff = pd.to_numeric(Z[effort_col], errors="coerce").fillna(0) <= 0
            if mask_zero_eff.any():
                Z.loc[mask_zero_eff, taxa_cols] = 0.0
        # build pools
        for pc, cols in self.pools_.items():
            if pc in self.selected_ and cols:
                Z[pc] = pd.to_numeric(Z[cols], errors="coerce").fillna(0).sum(axis=1)
        keep = list(dict.fromkeys(self.selected_ + self.group_cols_ + ([self.effort_col_] if self.effort_col_ else [])))
        return Z.reindex(columns=[c for c in keep if c in Z.columns])


In [12]:
# For all regions 1, 2, ..., 22
regiondfs = {}
for region in range(1, 23):
    print(f"Region: {region}")
    regiondf = df[df['HERlvl1Code'] == region]

    cleanregion = cleandf.IWANTMYXCLEAN(regiondf, thresh_high_missing=.90)
    regiondfs[region] = cleanregion

Region: 1
Dropped exact duplicate columns: ['Achac01', 'Achal01', 'Achca01', 'Achch01', 'Achcl01', 'Achco01', 'Achco03', 'Achcy01', 'Achde01', 'Achde02', 'Achdi01', 'Achdi02', 'Achel01', 'Achen01', 'Achex03', 'Achfl01', 'Achfr01', 'Achgr01', 'Achgr02', 'Achha01', 'Achhe01', 'Achhi01', 'Achho01', 'Achho02', 'Achim01', 'Achim02', 'Achim03', 'Achin01', 'Achin02', 'Achjo01', 'Achko01', 'Achkr01', 'Achkr03', 'Achkr04', 'Achku01', 'Achla01', 'Achla05', 'Achle01', 'Achle02', 'Achli02', 'Achli04', 'Achlo01', 'Achlu01', 'Achlu02', 'Achmo01', 'Achmo02', 'Achna01', 'Achna02', 'Achni01', 'Achno01', 'Achpa01', 'Achpe01', 'Achpe02', 'Achpl01', 'Achpo01', 'Achpr01', 'Achpr02', 'Achps01', 'Achps02', 'Achps03', 'Achpu01', 'Achro01', 'Achro03', 'Achru02', 'Achsc01', 'Achse01', 'Achse02', 'Achsi01', 'Achst01', 'Achst02', 'Achst04', 'Achsu02', 'Achsu04', 'Achtu01', 'Achzi01', 'Actde01', 'Actno01', 'Actro01', 'Actse01', 'Actsp01', 'Actsu01', 'Actvu01', 'Adlaq01', 'Adlbr01', 'Adlmu01', 'Adlmu02', 'Adlpa01',

In [13]:
# df = regiondfs[1]
# df

In [14]:
# X = df.drop(columns=[c for c in ["IBD","IBD_EQR","IBD_EQR_Status"] if c in df])
# y = df["IBD"].astype(float)

# reducer.fit(X, y)
# X_small = reducer.transform(X)

In [15]:
# reducer = FastSISReducer(SISConfig(
#     subsample_rows=20000,  # set 0 to use all
#     top_k=250,             # adjust budget
#     max_missing=0.98,
#     min_n_pairs=50,
#     corr_drop=0.98
# ))


In [16]:
# ============================================================
# MultiRegionBoostedClassifier (accuracy-focused) + selectable models
# + persistence + auto-visuals + per-region reports
# Patched: classifier wrapper (_estimator_type, decision_function) + safe calibration
# ============================================================

from typing import Dict, Any, List, Optional, Tuple
import warnings
import os, json, re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import joblib

from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, ClassifierMixin, clone, is_classifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier

# ---------- utilities ----------

def _slug(s) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "region"

def _normalize_model_names(models: Optional[List[str]]) -> Optional[set]:
    if not models:
        return None
    alias = {
        "HGB": "HGB",
        "HISTGRADIENTBOOSTING": "HGB",
        "LIGHTGBM": "LIGHTGBM", "LGBM": "LIGHTGBM",
        "XGBOOST": "XGBOOST", "XGB": "XGBOOST",
        "CATBOOST": "CATBOOST", "CAT": "CATBOOST",
    }
    out = set()
    for m in models:
        key = str(m).upper().replace(" ", "")
        out.add(alias.get(key, key))
    return out

def _resolve_col_case(df: pd.DataFrame, name: str) -> str:
    if name in df.columns:
        return name
    low = name.lower()
    for c in df.columns:
        if c.lower() == low:
            return c
    raise KeyError(f"Column '{name}' not found (case-insensitive).")

def _build_preprocessor(X: pd.DataFrame) -> Tuple[ColumnTransformer, List[str], List[str]]:
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="constant", fill_value="None")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), cat_cols),
        ],
        remainder="drop",
    )
    return pre, num_cols, cat_cols

def _feature_names_from_pre(pre: ColumnTransformer, num_cols: List[str], cat_cols: List[str]) -> List[str]:
    names: List[str] = []
    if num_cols:
        names.extend(num_cols)
    if cat_cols and "cat" in pre.named_transformers_:
        ohe = pre.named_transformers_["cat"].named_steps["ohe"]
        names.extend(ohe.get_feature_names_out(cat_cols).tolist())
    return names

def _cv_for(y: pd.Series, max_splits: int = 5) -> StratifiedKFold:
    min_class = int(y.value_counts(dropna=False).min())
    n_splits = max(2, min(max_splits, max(1, min_class)))
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def _can_kfold(y: pd.Series, cv: Optional[StratifiedKFold]) -> bool:
    if cv is None:
        return False
    return int(y.value_counts().min()) >= cv.get_n_splits()

# ---------- wrapper: contiguous labels per fit + fixed-width predict_proba ----------

class ContiguousLabelWrapper(BaseEstimator, ClassifierMixin):
    """
    Remap TRAIN labels to {0..k-1}. Predictions map back to global-encoded ints.
    If global_n_classes_ is set, predict_proba returns fixed K columns in global-id order.
    """
    _estimator_type = "classifier"  # ensure sklearn treats this as a classifier

    def __init__(self, estimator: BaseEstimator, global_n_classes_: Optional[int] = None):
        self.estimator = estimator
        self.global_n_classes_ = global_n_classes_
        self.estimator_ = None
        self.classes_ = None  # global int labels present in this fit

    def get_params(self, deep=True):
        params = {"estimator": self.estimator, "global_n_classes_": self.global_n_classes_}
        if deep and hasattr(self.estimator, "get_params"):
            for k, v in self.estimator.get_params(deep=True).items():
                params[f"estimator__{k}"] = v
        return params

    def set_params(self, **params):
        est_params = {}
        for k, v in list(params.items()):
            if k == "estimator":
                self.estimator = v
            elif k == "global_n_classes_":
                self.global_n_classes_ = v
            elif k.startswith("estimator__"):
                est_params[k.split("__", 1)[1]] = v
            else:
                setattr(self, k, v)
            params.pop(k, None)
        if est_params and hasattr(self.estimator, "set_params"):
            self.estimator.set_params(**est_params)
        return self

    def fit(self, X, y, **fit_params):
        y = np.asarray(y)
        uniq = np.unique(y)                 # global-encoded ints present in TRAIN
        to_local = {g: i for i, g in enumerate(uniq)}
        y_local = np.array([to_local[val] for val in y], dtype=int)
        est = clone(self.estimator)
        est.fit(X, y_local, **fit_params)
        self.estimator_ = est
        self.classes_ = uniq
        return self

    def predict(self, X):
        y_local = self.estimator_.predict(X)
        return self.classes_[np.asarray(y_local, dtype=int)]

    def predict_proba(self, X):
        if not hasattr(self.estimator_, "predict_proba"):
            raise AttributeError("Underlying estimator has no predict_proba.")
        P_local = self.estimator_.predict_proba(X)  # (n, k_local)
        if self.global_n_classes_ is None:
            return P_local
        n, k_glob = P_local.shape[0], int(self.global_n_classes_)
        P = np.zeros((n, k_glob), dtype=float)
        for j, g in enumerate(self.classes_):
            if 0 <= int(g) < k_glob:
                P[:, int(g)] = P_local[:, j]
        row_sums = P.sum(axis=1, keepdims=True)
        fix = (row_sums == 0).flatten()
        if np.any(fix):
            P[fix, :] = 1.0 / k_glob
        else:
            P /= np.where(row_sums == 0, 1, row_sums)
        return P

    def decision_function(self, X):
        if hasattr(self.estimator_, "decision_function"):
            return self.estimator_.decision_function(X)
        raise AttributeError("Underlying estimator has no decision_function.")

# ---------- main class (accuracy-focused boosted selection) ----------

class MultiRegionBoostedClassifier:
    """
    Per-region selection among boosted trees with label remapping and visuals:
      - HGB, LightGBM*, XGBoost*, CatBoost* (*if installed)
    Select models via constructor or per-call `models=[...]` (e.g., ["HGB","XGBoost"]).
    """

    def __init__(self, xcleans: Dict[Any, pd.DataFrame], random_state: int = 42,
                 verbose: bool = True, models: Optional[List[str]] = None):
        self.xcleans = xcleans
        self.random_state = random_state
        self.verbose = verbose
        self.results: Dict[Any, Dict[str, Any]] = {}
        self.target_column: Optional[str] = None
        self.models_allowed = _normalize_model_names(models)  # None => all available

    def _candidate_spaces(self, n_classes: Optional[int], allowed: Optional[set]) -> Dict[str, Tuple[Any, dict]]:
        rng = self.random_state
        spaces: Dict[str, Tuple[Any, dict]] = {}

        def ok(tag: str) -> bool:
            return allowed is None or tag in allowed

        # HGB
        if ok("HGB"):
            hgb = ContiguousLabelWrapper(
                HistGradientBoostingClassifier(random_state=rng),
                global_n_classes_=n_classes
            )
            hgb_grid = {
                "model__estimator__learning_rate": loguniform(1e-2, 3e-1),
                "model__estimator__max_depth": randint(2, 13),
                "model__estimator__max_leaf_nodes": randint(16, 257),
                "model__estimator__min_samples_leaf": randint(10, 201),
                "model__estimator__l2_regularization": loguniform(1e-4, 10.0),
                "model__estimator__max_bins": randint(64, 257),
            }
            spaces["HGB"] = (hgb, hgb_grid)

        # LightGBM
        if ok("LIGHTGBM"):
            try:
                from lightgbm import LGBMClassifier
                lgb = ContiguousLabelWrapper(
                    LGBMClassifier(objective="multiclass", random_state=rng, n_jobs=-1, verbose=-1),
                    global_n_classes_=n_classes
                )
                lgb_grid = {
                    "model__estimator__n_estimators": randint(300, 1201),
                    "model__estimator__learning_rate": loguniform(1e-2, 3e-1),
                    "model__estimator__num_leaves": randint(16, 257),
                    "model__estimator__max_depth": randint(-1, 16),
                    "model__estimator__min_child_samples": randint(5, 201),
                    "model__estimator__subsample": uniform(0.5, 0.5),
                    "model__estimator__colsample_bytree": uniform(0.5, 0.5),
                    "model__estimator__reg_alpha": loguniform(1e-6, 10.0),
                    "model__estimator__reg_lambda": loguniform(1e-6, 10.0),
                }
                spaces["LightGBM"] = (lgb, lgb_grid)
            except Exception:
                if self.verbose: print("[warn] LightGBM not available; skipping.")

        # XGBoost
        if ok("XGBOOST"):
            try:
                from xgboost import XGBClassifier
                xgb = ContiguousLabelWrapper(
                    XGBClassifier(objective="multi:softprob", random_state=rng, tree_method="hist",
                                  eval_metric="mlogloss", n_jobs=-1, verbosity=0),
                    global_n_classes_=n_classes
                )
                xgb_grid = {
                    "model__estimator__n_estimators": randint(300, 1201),
                    "model__estimator__learning_rate": loguniform(1e-2, 3e-1),
                    "model__estimator__max_depth": randint(3, 13),
                    "model__estimator__min_child_weight": loguniform(1e-1, 10.0),
                    "model__estimator__subsample": uniform(0.5, 0.5),
                    "model__estimator__colsample_bytree": uniform(0.5, 0.5),
                    "model__estimator__gamma": loguniform(1e-6, 10.0),
                    "model__estimator__reg_alpha": loguniform(1e-6, 10.0),
                    "model__estimator__reg_lambda": loguniform(1e-6, 10.0),
                }
                spaces["XGBoost"] = (xgb, xgb_grid)
            except Exception:
                if self.verbose: print("[warn] XGBoost not available; skipping.")

        # CatBoost
        if ok("CATBOOST"):
            try:
                from catboost import CatBoostClassifier
                cb = ContiguousLabelWrapper(
                    CatBoostClassifier(loss_function="MultiClass", random_seed=rng,
                                       verbose=False, allow_writing_files=False, thread_count=-1),
                    global_n_classes_=n_classes
                )
                cb_grid = {
                    "model__estimator__iterations": randint(300, 1201),
                    "model__estimator__learning_rate": loguniform(1e-2, 3e-1),
                    "model__estimator__depth": randint(4, 11),
                    "model__estimator__l2_leaf_reg": loguniform(1e-3, 30.0),
                    "model__estimator__bagging_temperature": loguniform(1e-2, 10.0),
                }
                spaces["CatBoost"] = (cb, cb_grid)
            except Exception:
                if self.verbose: print("[warn] CatBoost not available; skipping.")

        return spaces

    def process_missing_class(
        self,
        target_column: str = "IBD_EQR_Status",
        max_splits: int = 5,
        n_iter_per_model: int = 25,
        scoring: str = "accuracy",
        calibrate: bool = True,
        prob_method: str = "predict_proba",
        models: Optional[List[str]] = None,
    ):
        self.results.clear()
        self.target_column = target_column

        for key, Xclean in self.xcleans.items():
            try:
                y_col = _resolve_col_case(Xclean, target_column)
            except KeyError:
                if self.verbose: print(f"[skip {key}] missing target '{target_column}'"); continue
            if len(Xclean) < 5:
                if self.verbose: print(f"[skip {key}] too few rows: {len(Xclean)}"); continue

            y = Xclean[y_col]
            leak = {y_col, "IBD", "IBD_EQR", "IBD_EQR_Status", "IBD_EQR_STATUS"}
            X = Xclean.drop(columns=[c for c in leak if c in Xclean.columns], errors="ignore")

            mask_tr = y.notna()
            mask_te = ~mask_tr
            if mask_tr.sum() == 0:
                if self.verbose: print(f"[skip {key}] no labeled rows"); continue

            le = LabelEncoder()
            y_tr = y.loc[mask_tr]
            y_tr_enc = pd.Series(le.fit_transform(y_tr), index=y_tr.index)
            K = len(le.classes_)

            pre, num_cols, cat_cols = _build_preprocessor(X)
            cv = _cv_for(y_tr, max_splits=max_splits)

            allowed = _normalize_model_names(models) or self.models_allowed
            candidates = self._candidate_spaces(n_classes=K, allowed=allowed)
            if not candidates:
                raise RuntimeError("No boosted candidates available with the given 'models' setting.")

            best = {"name": None, "estimator": None, "cv_score": -np.inf, "cv_results_": None}
            for name, (estimator, grid) in candidates.items():
                pipe = Pipeline([("pre", pre), ("model", estimator)])
                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=grid,
                    n_iter=n_iter_per_model,
                    scoring=scoring,
                    cv=cv,
                    refit=True,
                    n_jobs=-1,
                    verbose=0,
                    random_state=self.random_state,
                    error_score=np.nan,
                )
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    search.fit(X.loc[mask_tr], y_tr_enc)

                mean_score = np.nanmax(search.cv_results_["mean_test_score"])
                if self.verbose:
                    print(f"[{key}] {name}: best {scoring}={mean_score:.3f}")
                if np.isfinite(mean_score) and mean_score > best["cv_score"]:
                    best.update({"name": name, "estimator": search.best_estimator_, "cv_score": float(mean_score),
                                 "cv_results_": search.cv_results_})

            if best["estimator"] is None:
                if self.verbose: print(f"[skip {key}] no model selected")
                continue

            best_pipe: Pipeline = best["estimator"]

            # OOF diagnostics
            have_proba = False
            oof_proba = None
            if _can_kfold(y_tr, cv):
                try:
                    oof_pred_enc = cross_val_predict(best_pipe, X.loc[mask_tr], y_tr_enc, cv=cv, method="predict", n_jobs=-1)
                except Exception:
                    oof_pred_enc = cross_val_predict(best_pipe, X.loc[mask_tr], y_tr_enc, cv=cv, method="predict", n_jobs=-1)
                try:
                    oof_proba = cross_val_predict(best_pipe, X.loc[mask_tr], y_tr_enc, cv=cv, method="predict_proba", n_jobs=-1)
                    have_proba = True
                except Exception:
                    have_proba = False
                oof_pred = pd.Series(le.inverse_transform(oof_pred_enc.astype(int)), index=y_tr.index)
                oof_acc = accuracy_score(y_tr, oof_pred)
                oof_f1_macro = f1_score(y_tr, oof_pred, average="macro")
                cm = confusion_matrix(y_tr, oof_pred, labels=le.classes_)
            else:
                if self.verbose:
                    print(f"[{key}] skip OOF: min_class={int(y_tr.value_counts().min())} < folds={cv.get_n_splits()}")
                best_pipe.fit(X.loc[mask_tr], y_tr_enc)
                oof_pred_enc = best_pipe.predict(X.loc[mask_tr])
                oof_pred = pd.Series(le.inverse_transform(oof_pred_enc.astype(int)), index=y_tr.index)
                if hasattr(best_pipe, "predict_proba"):
                    try:
                        oof_proba = best_pipe.predict_proba(X.loc[mask_tr])
                        have_proba = True
                    except Exception:
                        have_proba = False
                oof_acc = accuracy_score(y_tr, oof_pred)
                oof_f1_macro = f1_score(y_tr, oof_pred, average="macro")
                cm = confusion_matrix(y_tr, oof_pred, labels=le.classes_)

            cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in le.classes_],
                                 columns=[f"pred_{c}" for c in le.classes_])

            # Refit on all labeled rows
            best_pipe.fit(X.loc[mask_tr], y_tr_enc)

            # Calibration (safe)
            calibrated_pipe = best_pipe
            if calibrate and _can_kfold(y_tr, cv):
                base = best_pipe
                final_est = base.named_steps["model"]
                if is_classifier(final_est) and (hasattr(final_est, "predict_proba") or hasattr(final_est, "decision_function")):
                    calib = CalibratedClassifierCV(final_est, cv=cv, method="isotonic")
                    calibrated_pipe = Pipeline([("pre", base.named_steps["pre"]), ("model", calib)])
                    calibrated_pipe.fit(X.loc[mask_tr], y_tr_enc)
                else:
                    if self.verbose: print(f"[{key}] skip calibration: estimator lacks proba/decision_function")
            else:
                if calibrate and self.verbose:
                    print(f"[{key}] skip calibration: min_class={int(y_tr.value_counts().min())} < folds={cv.get_n_splits()}")

            # Predict missing rows
            if mask_te.any():
                X_te = X.loc[mask_te]
                yhat_enc = calibrated_pipe.predict(X_te)
                yhat_te = pd.Series(le.inverse_transform(yhat_enc.astype(int)), index=X_te.index, name="yhat_te")

                proba_te = pd.DataFrame(index=X_te.index)
                if hasattr(calibrated_pipe, "predict_proba"):
                    P = calibrated_pipe.predict_proba(X_te)
                    inner = calibrated_pipe.named_steps["model"]
                    classes_enc = getattr(inner, "classes_", None)
                    if classes_enc is None and hasattr(inner, "base_estimator"):
                        classes_enc = getattr(inner.base_estimator, "classes_", None)
                    if classes_enc is None:
                        try:
                            classes_enc = inner.estimators_[0].classes_
                        except Exception:
                            classes_enc = np.arange(P.shape[1])
                    classes = le.inverse_transform(np.array(classes_enc, dtype=int))
                    proba_te = pd.DataFrame(P, index=X_te.index, columns=[f"proba_{c}" for c in classes])
            else:
                yhat_te = pd.Series(dtype=object, name="yhat_te")
                proba_te = pd.DataFrame()

            # Feature importances (if available)
            final_est = best_pipe.named_steps["model"]
            pre_fitted = best_pipe.named_steps["pre"]
            feat_names = _feature_names_from_pre(pre_fitted, num_cols, cat_cols)
            fi = pd.Series(dtype=float)
            try:
                est_inner = getattr(final_est, "estimator_", None)
                if est_inner is None and hasattr(final_est, "estimator"):
                    est_inner = final_est.estimator
                if est_inner is not None and hasattr(est_inner, "feature_importances_"):
                    fi = pd.Series(est_inner.feature_importances_, index=feat_names).sort_values(ascending=False)
            except Exception:
                pass

            # Filled target
            y_filled = y.copy()
            if mask_te.any():
                y_filled.loc[mask_te] = yhat_te.values

            # Store
            res = {
                "best_name": best["name"],
                "best_cv_score_acc": float(best["cv_score"]),
                "oof_acc": oof_acc,
                "oof_f1_macro": oof_f1_macro,
                "oof_confusion": cm_df,
                "classes_": le.classes_,
                "label_encoder": le,
                "model": calibrated_pipe,
                "uncalibrated_model": best_pipe,
                "n_train": int(mask_tr.sum()),
                "n_infer": int(mask_te.sum()),
                "y_tr": y_tr,
                "y_tr_enc": y_tr_enc,
                "oof_pred": pd.Series(oof_pred, index=y_tr.index, name="oof_pred"),
                "oof_proba": (oof_proba if have_proba else None),
                "proba_available": bool(have_proba),
                "yhat_te": yhat_te,
                "proba_te": proba_te,
                "y_filled": y_filled,
                "feature_importances": fi,
                "cv_results_": best.get("cv_results_", None),
            }
            # Back-compat aliases
            res["best_cv_score_bal_acc"] = res["best_cv_score_acc"]
            res["oof_bal_acc"] = res["oof_acc"]

            self.results[key] = res

            if self.verbose:
                print(f"[done {key}] best={best['name']}  OOF_acc={oof_acc:.3f}  "
                      f"OOF_f1_macro={oof_f1_macro:.3f}  infer={int(mask_te.sum())}")

    # ---- convenience ----

    def infer_table(self, key: Any) -> pd.DataFrame:
        r = self.results[key]
        idx = self.xcleans[key].index
        out = pd.DataFrame(index=idx)
        out["y_true"] = r["y_tr"]
        out["oof_pred"] = r["oof_pred"]
        out["yhat_te"] = r["yhat_te"]
        out["y_filled"] = r["y_filled"]
        out = out.join(r["proba_te"], how="left")
        return out

    def all_inferred(self) -> pd.DataFrame:
        frames = []
        for k in self.results:
            df = self.infer_table(k).reset_index().rename(columns={"index": "row_id"})
            df.insert(0, "region", k)
            frames.append(df)
        return pd.concat(frames, ignore_index=True)

    def filled_dataframe(self, key: Any) -> pd.DataFrame:
        if self.target_column is None:
            raise RuntimeError("Run process_missing_class first.")
        df = self.xcleans[key].copy()
        col = _resolve_col_case(df, self.target_column)
        df[col] = self.results[key]["y_filled"]
        return df

    def top_features(self, key: Any, k: int = 20) -> pd.DataFrame:
        fi = self.results[key]["feature_importances"]
        if fi.empty:
            raise ValueError("Feature importances unavailable for the selected model.")
        return fi.head(k).reset_index().rename(columns={"index": "feature", 0: "importance"})

    def print_report(self, key: Any):
        r = self.results[key]
        y_tr = r["y_tr"]
        y_pred = r["oof_pred"].reindex(y_tr.index)
        print(classification_report(y_tr, y_pred, digits=3))
        print("OOF confusion matrix:")
        print(r["oof_confusion"])

    def plot_oof_confusion(self, key: Any, savepath: Optional[str] = None, normalize: Optional[str] = None):
        cm = self.results[key]["oof_confusion"].copy()
        if normalize == "row":
            cm = cm.div(cm.sum(axis=1).replace(0, 1), axis=0)
            title = f"OOF confusion (row-norm) — region {key}"
        elif normalize == "col":
            cm = cm.div(cm.sum(axis=0).replace(0, 1), axis=1)
            title = f"OOF confusion (col-norm) — region {key}"
        else:
            title = f"OOF confusion — region {key}"
        plt.figure(figsize=(5, 4))
        plt.imshow(cm.values, interpolation="nearest")
        plt.xticks(range(len(cm.columns)), cm.columns, rotation=45, ha="right")
        plt.yticks(range(len(cm.index)), cm.index)
        plt.title(title)
        plt.colorbar()
        plt.tight_layout()
        if savepath:
            Path(savepath).parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(savepath, dpi=150)
            plt.close()
        else:
            plt.show()

    def predict_new(self, key: Any, X_new: pd.DataFrame) -> np.ndarray:
        return self.results[key]["model"].predict(X_new)

    def predict_labels_new(self, key: Any, X_new: pd.DataFrame) -> np.ndarray:
        enc = self.results[key]["model"].predict(X_new)
        le: LabelEncoder = self.results[key]["label_encoder"]
        return le.inverse_transform(enc.astype(int))

    def predict_proba_new(self, key: Any, X_new: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        mdl = self.results[key]["model"]
        if not hasattr(mdl, "predict_proba"):
            raise AttributeError("Model does not support predict_proba.")
        P = mdl.predict_proba(X_new)
        inner = mdl.named_steps["model"]
        classes_enc = getattr(inner, "classes_", None)
        if classes_enc is None:
            try:
                classes_enc = inner.estimators_[0].classes_
            except Exception:
                classes_enc = np.arange(P.shape[1])
        le: LabelEncoder = self.results[key]["label_encoder"]
        classes = le.inverse_transform(np.array(classes_enc, dtype=int))
        return P, classes

    # ---------- visuals helpers ----------

    @staticmethod
    def _savefig(path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        plt.tight_layout()
        plt.savefig(path, dpi=150)
        plt.close()

    def _export_visuals(self, key: Any, r: Dict[str, Any], viz_dir: Path, top_k_classes: int = 6, top_k_feats: int = 20):
        viz_dir.mkdir(parents=True, exist_ok=True)
        # 1) Confusion matrix (counts)
        cm = r["oof_confusion"]
        plt.figure(figsize=(6, 5))
        plt.imshow(cm.values, interpolation="nearest")
        plt.xticks(range(len(cm.columns)), cm.columns, rotation=45, ha="right")
        plt.yticks(range(len(cm.index)), cm.index)
        plt.title(f"OOF Confusion — {key}")
        plt.colorbar()
        self._savefig(viz_dir / "confusion_counts.png")

        # 1b) Confusion matrix (row-normalized)
        cm_norm = cm.div(cm.sum(axis=1).replace(0, 1), axis=0)
        plt.figure(figsize=(6, 5))
        plt.imshow(cm_norm.values, interpolation="nearest")
        plt.xticks(range(len(cm_norm.columns)), cm_norm.columns, rotation=45, ha="right")
        plt.yticks(range(len(cm_norm.index)), cm_norm.index)
        plt.title(f"OOF Confusion (row-normalized) — {key}")
        plt.colorbar()
        self._savefig(viz_dir / "confusion_row_norm.png")

        # 2) Classification report table -> CSV and F1 bar
        y_tr = r["y_tr"]
        y_pred = r["oof_pred"].reindex(y_tr.index)
        rep = classification_report(y_tr, y_pred, output_dict=True, zero_division=0)
        pd.DataFrame(rep).to_csv(viz_dir / "classification_report.csv", index=True)
        cls = [c for c in rep.keys() if c not in {"accuracy", "macro avg", "weighted avg"}]
        f1_vals = [rep[c]["f1-score"] for c in cls]
        plt.figure(figsize=(max(6, 0.4*len(cls)), 3.6))
        ax = plt.gca()
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        plt.bar(cls, f1_vals)
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("F1-score")
        plt.title(f"Per-class F1 — {key}")
        self._savefig(viz_dir / "per_class_f1.png")

        # 3) ROC and PR (if probabilities available)
        if r.get("proba_available", False) and r["oof_proba"] is not None:
            P = np.asarray(r["oof_proba"])
            classes = r["classes_"]
            y_bin = label_binarize(y_tr, classes=classes)
            if P.shape[1] == len(classes) and y_bin.shape[0] == P.shape[0]:
                try:
                    auc_macro = roc_auc_score(y_bin, P, average="macro", multi_class="ovr")
                except Exception:
                    auc_macro = np.nan
                try:
                    ap_scores = [average_precision_score(y_bin[:, j], P[:, j]) for j in range(P.shape[1])]
                    ap_macro = float(np.nanmean(ap_scores))
                except Exception:
                    ap_macro = np.nan
                with open(viz_dir / "prob_metrics.json", "w", encoding="utf-8") as f:
                    json.dump({"roc_auc_macro_ovr": auc_macro, "avg_precision_macro": ap_macro}, f, indent=2)

                support = y_tr.value_counts().reindex(classes, fill_value=0)
                top_idx = np.argsort(-support.values)[:min(top_k_classes, len(classes))]

                plt.figure(figsize=(6, 5))
                for j in top_idx:
                    fpr, tpr, _ = roc_curve(y_bin[:, j], P[:, j])
                    plt.plot(fpr, tpr, label=f"ROC {classes[j]}")
                plt.plot([0, 1], [0, 1], linestyle="--")
                plt.xlabel("FPR"); plt.ylabel("TPR")
                plt.title(f"ROC OVR (top-{len(top_idx)}) — {key}")
                plt.legend(fontsize=8)
                self._savefig(viz_dir / "roc_ovr_topk.png")

                plt.figure(figsize=(6, 5))
                for j in top_idx:
                    pr, rc, _ = precision_recall_curve(y_bin[:, j], P[:, j])
                    plt.plot(rc, pr, label=f"PR {classes[j]}")
                plt.xlabel("Recall"); plt.ylabel("Precision")
                plt.title(f"PR OVR (top-{len(top_idx)}) — {key}")
                plt.legend(fontsize=8)
                self._savefig(viz_dir / "pr_ovr_topk.png")

                plt.figure(figsize=(6, 5))
                for j in top_idx:
                    prob_true, prob_pred = calibration_curve(y_bin[:, j], P[:, j], n_bins=10, strategy="uniform")
                    plt.plot(prob_pred, prob_true, marker="o", label=f"Cal {classes[j]}")
                plt.plot([0, 1], [0, 1], linestyle="--")
                plt.xlabel("Predicted probability"); plt.ylabel("Observed frequency")
                plt.title(f"Reliability (top-{len(top_idx)}) — {key}")
                plt.legend(fontsize=8)
                self._savefig(viz_dir / "calibration_topk.png")

        # 4) Feature importances
        fi = r["feature_importances"]
        if isinstance(fi, pd.Series) and not fi.empty:
            k = min(top_k_feats, len(fi))
            plt.figure(figsize=(8, max(3.5, 0.25*k)))
            fi.head(k)[::-1].plot(kind="barh")
            plt.xlabel("Importance")
            plt.title(f"Top-{k} feature importances — {key}")
            self._savefig(viz_dir / "feature_importances_topk.png")

        # 5) CV score trace
        cvres = r.get("cv_results_", None)
        if cvres is not None:
            dfcv = pd.DataFrame(cvres)
            dfcv.to_csv(viz_dir / "cv_results.csv", index=False)
            if "mean_test_score" in dfcv:
                plt.figure(figsize=(6, 4))
                plt.plot(np.arange(len(dfcv["mean_test_score"])), dfcv["mean_test_score"], marker="o", linestyle="-")
                plt.xlabel("Candidate #"); plt.ylabel("mean_test_score")
                plt.title(f"RandomizedSearch scores — {key}")
                self._savefig(viz_dir / "cv_scores.png")

        # 6) Save compact metrics JSON
        metrics = {
            "oof_acc": float(r["oof_acc"]),
            "oof_f1_macro": float(r["oof_f1_macro"]),
            "n_train": int(r["n_train"]),
            "n_infer": int(r["n_infer"]),
        }
        with open(viz_dir / "metrics_summary.json", "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

    # ---------- persistence ----------

    def save_all(self, outdir: str = "models", compress: int = 3, make_figs: bool = True,
                 figs_folder_suffix: str = "_viz", top_k_classes: int = 6, top_k_feats: int = 20) -> pd.DataFrame:
        Path(outdir).mkdir(parents=True, exist_ok=True)
        rows = []
        for key, r in self.results.items():
            best_name = (r["best_name"] or "model").lower()
            base = Path(outdir) / f"best_{best_name}_{_slug(key)}"

            joblib.dump(r["model"], f"{base}.joblib", compress=compress)
            joblib.dump(r["uncalibrated_model"], f"{base}_uncal.joblib", compress=compress)
            joblib.dump(r["label_encoder"], f"{base}_label_encoder.joblib", compress=compress)

            scores = {
                "best_cv_score_acc": r.get("best_cv_score_acc", r.get("best_cv_score_bal_acc")),
                "oof_acc": r.get("oof_acc", r.get("oof_bal_acc")),
                "oof_f1_macro": r["oof_f1_macro"],
            }
            meta = {
                "region": str(key),
                "best_name": r["best_name"],
                "paths": {
                    "calibrated": f"{base}.joblib",
                    "uncalibrated": f"{base}_uncal.joblib",
                    "label_encoder": f"{base}_label_encoder.joblib",
                    "meta": f"{base}_meta.json",
                },
                "scores": scores,
                "classes": list(map(str, r["classes_"])),
                "n_train": r["n_train"],
                "n_infer": r["n_infer"],
                "saved_at": datetime.now().isoformat(timespec="seconds"),
                "target_column": self.target_column,
            }
            with open(f"{base}_meta.json", "w", encoding="utf-8") as f:
                json.dump(meta, f, indent=2, ensure_ascii=False)

            if make_figs:
                viz_dir = Path(str(base) + figs_folder_suffix)
                try:
                    self._export_visuals(key, r, viz_dir=viz_dir, top_k_classes=top_k_classes, top_k_feats=top_k_feats)
                except Exception as e:
                    if self.verbose:
                        print(f"[warn] visuals for {key} failed: {e}")

            rows.append({
                "region": key,
                "best_name": r["best_name"],
                "calibrated_path": f"{base}.joblib",
                "encoder_path": f"{base}_label_encoder.joblib",
                "meta_path": f"{base}_meta.json",
                "viz_dir": (str(Path(str(base) + figs_folder_suffix)) if make_figs else None),
                "oof_acc": scores["oof_acc"],
                "oof_f1_macro": scores["oof_f1_macro"],
            })
        return pd.DataFrame(rows)

    @staticmethod
    def load_bundle(basepath: str):
        pipe = joblib.load(basepath + ".joblib")
        le = joblib.load(basepath + "_label_encoder.joblib")
        with open(basepath + "_meta.json", "r", encoding="utf-8") as f:
            meta = json.load(f)
        return pipe, le, meta

    # ---------- report helpers ----------

    def save_oof_report(self, key: Any, outdir: str = "models",
                        figs_folder_suffix: str = "_viz",
                        top_k_classes: int = 6, top_k_feats: int = 20) -> str:
        if key not in self.results:
            raise KeyError(f"Unknown region key: {key}")
        r = self.results[key]
        best_name = (r["best_name"] or "model").lower()
        base = Path(outdir) / f"best_{best_name}_{_slug(key)}"
        viz_dir = Path(str(base) + figs_folder_suffix)
        viz_dir.mkdir(parents=True, exist_ok=True)

        self._export_visuals(key, r, viz_dir=viz_dir,
                             top_k_classes=top_k_classes, top_k_feats=top_k_feats)

        r["oof_confusion"].to_csv(viz_dir / "confusion_matrix.csv")
        y_tr = r["y_tr"]
        y_pred = r["oof_pred"].reindex(y_tr.index)
        with open(viz_dir / "classification_report.txt", "w", encoding="utf-8") as f:
            f.write(classification_report(y_tr, y_pred, digits=4, zero_division=0))

        df = pd.DataFrame({"y_true": y_tr, "oof_pred": y_pred})
        if r.get("proba_available", False) and r["oof_proba"] is not None:
            P = np.asarray(r["oof_proba"])
            classes = r["classes_"]
            for j, c in enumerate(classes):
                df[f"proba_{c}"] = P[:, j]
        df.to_csv(viz_dir / "oof_predictions.csv", index=True)

        manifest = {
            "region": str(key),
            "best_name": r["best_name"],
            "report_dir": str(viz_dir),
            "n_train": int(r["n_train"]),
            "n_infer": int(r["n_infer"]),
            "oof_acc": float(r["oof_acc"]),
            "oof_f1_macro": float(r["oof_f1_macro"]),
        }
        with open(viz_dir / "manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)
        return str(viz_dir)

    def save_all_reports(self, outdir: str = "models",
                         figs_folder_suffix: str = "_viz",
                         top_k_classes: int = 6, top_k_feats: int = 20) -> pd.DataFrame:
        rows = []
        for key in self.results:
            path = self.save_oof_report(key, outdir=outdir,
                                        figs_folder_suffix=figs_folder_suffix,
                                        top_k_classes=top_k_classes, top_k_feats=top_k_feats)
            r = self.results[key]
            rows.append({
                "region": key,
                "best_name": r["best_name"],
                "report_dir": path,
                "oof_acc": float(r["oof_acc"]),
                "oof_f1_macro": float(r["oof_f1_macro"]),
            })
        return pd.DataFrame(rows)

# ----------------- usage example -----------------
# xcleans = {"region1": df1, "region2": df2, ...}
# # Skip CatBoost globally:
# auto = MultiRegionBoostedClassifier(xcleans, random_state=42, verbose=True,
#                                     models=["HGB", "LightGBM", "XGBoost"])
# auto.process_missing_class(scoring="accuracy", n_iter_per_model=25, max_splits=5, calibrate=True)
# manifest = auto.save_all(outdir="models", compress=3, make_figs=True)
# reports = auto.save_all_reports(outdir="models")
# print(manifest)
# print(reports)


In [17]:
auto = MultiRegionBoostedClassifier(regiondfs, random_state=42, verbose=True, models=["CatBoost"])
auto.process_missing_class(scoring="accuracy", n_iter_per_model=3, max_splits=5, calibrate=True)

[1] CatBoost: best accuracy=0.860
[1] skip OOF: min_class=1 < folds=2


c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\preprocessing\_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[1] skip calibration: min_class=1 < folds=2
[done 1] best=CatBoost  OOF_acc=1.000  OOF_f1_macro=1.000  infer=172


c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\preprocessing\_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[2] CatBoost: best accuracy=0.922
[2] skip OOF: min_class=1 < folds=2


...

c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\preprocessing\_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[22] CatBoost: best accuracy=0.675


c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\preprocessing\_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[22] skip calibration: estimator lacks proba/decision_function
[done 22] best=CatBoost  OOF_acc=0.675  OOF_f1_macro=0.504  infer=32


c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\preprocessing\_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [18]:
manifest = auto.save_all(outdir="models", compress=3, make_figs=True)
manifest

,region,best_name,calibrated_path,encoder_path,meta_path,viz_dir,oof_acc,oof_f1_macro
0,1,CatBoost,models\best_catboost_1.joblib,models\best_catboost_1_label_encoder.joblib,models\best_catboost_1_meta.json,models\best_catboost_1_viz,1.000000,1.000000
1,2,CatBoost,models\best_catboost_2.joblib,models\best_catboost_2_label_encoder.joblib,models\best_catboost_2_meta.json,models\best_catboost_2_viz,1.000000,1.000000
2,3,CatBoost,models\best_catboost_3.joblib,models\best_catboost_3_label_encoder.joblib,models\best_catboost_3_meta.json,models\best_catboost_3_viz,0.813318,0.741343
3,4,CatBoost,models\best_catboost_4.joblib,models\best_catboost_4_label_encoder.joblib,models\best_catboost_4_meta.json,models\best_catboost_4_viz,0.704453,0.611561
4,5,CatBoost,models\best_catboost_5.joblib,models\best_catboost_5_label_encoder.joblib,models\best_catboost_5_meta.json,models\best_catboost_5_viz,0.859836,0.749431
5,6,CatBoost,models\best_catboost_6.joblib,models\best_catboost_6_label_encoder.joblib,models\best_catboost_6_meta.json,models\best_catboost_6_viz,0.827187,0.732432
6,7,CatBoost,models\best_catboost_7.joblib,models\best_catboost_7_label_encoder.joblib,models\best_catboost_7_meta.json,models\best_catboost_7_viz,0.886266,0.714921
7,8,CatBoost,models\best_catboost_8.joblib,models\best_catboost_8_label_encoder.joblib,models\best_catboost_8_meta.json,models\best_catboost_8_viz,1.000000,1.000000
8,9,CatBoost,models\best_catboost_9.joblib,models\best_catboost_9_label_encoder.joblib,models\best_catboost_9_meta.json,models\best_catboost_9_viz,0.883538,0.631213
9,10,CatBoost,models\best_catboost_10.joblib,models\best_catboost_10_label_encoder.joblib,models\best_catboost_10_meta.json,models\best_catboost_10_viz,0.848628,0.709646


In [19]:
# all regions
reports = auto.save_all_reports(outdir="models")
reports

,region,best_name,report_dir,oof_acc,oof_f1_macro
0,1,CatBoost,models\best_catboost_1_viz,1.000000,1.000000
1,2,CatBoost,models\best_catboost_2_viz,1.000000,1.000000
2,3,CatBoost,models\best_catboost_3_viz,0.813318,0.741343
3,4,CatBoost,models\best_catboost_4_viz,0.704453,0.611561
4,5,CatBoost,models\best_catboost_5_viz,0.859836,0.749431
5,6,CatBoost,models\best_catboost_6_viz,0.827187,0.732432
6,7,CatBoost,models\best_catboost_7_viz,0.886266,0.714921
7,8,CatBoost,models\best_catboost_8_viz,1.000000,1.000000
8,9,CatBoost,models\best_catboost_9_viz,0.883538,0.631213
9,10,CatBoost,models\best_catboost_10_viz,0.848628,0.709646


In [20]:
all_preds = auto.all_inferred()
all_preds.to_csv("all_regions_inferred.csv", index=False)

# reportes

In [21]:
for i in range(3, 23):
    print(f"\n Region {i} report:")
    auto.print_report(i)
    print("\n")


 Region 3 report:
              precision    recall  f1-score   support

         Bad      0.711     0.432     0.538        74
        Good      0.768     0.769     0.768      1249
        High      0.880     0.870     0.875      1180
    Moderate      0.815     0.875     0.844      1522
        Poor      0.756     0.620     0.681       405

    accuracy                          0.813      4430
   macro avg      0.786     0.713     0.741      4430
weighted avg      0.812     0.813     0.811      4430

OOF confusion matrix:
               pred_Bad  pred_Good  pred_High  pred_Moderate  pred_Poor
true_Bad             32          0          0              1         41
true_Good             0        961        133            155          0
true_High             0        148       1027              5          0
true_Moderate         1        142          7           1332         40
true_Poor            12          1          0            141        251



 Region 4 report:
              pre

c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\175199\.conda\envs\a\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\1751

# results

In [22]:
auto.results

{1: {'best_name': 'CatBoost',
  'best_cv_score_acc': 0.8598628280803355,
  'oof_acc': 1.0,
  'oof_f1_macro': 1.0,
  'oof_confusion':                pred_Bad  pred_Good  pred_High  pred_Moderate  pred_Poor
  true_Bad              1          0          0              0          0
  true_Good             0        232          0              0          0
  true_High             0          0        988              0          0
  true_Moderate         0          0          0             86          0
  true_Poor             0          0          0              0          6,
  'classes_': array(['Bad', 'Good', 'High', 'Moderate', 'Poor'], dtype=object),
  'label_encoder': LabelEncoder(),
  'model': Pipeline(steps=[('pre',
                   ColumnTransformer(transformers=[('num',
                                                    Pipeline(steps=[('imp',
                                                                     SimpleImputer(strategy='median'))]),
                                 

In [23]:
for i in range(3, 23):
    print(f"Region {i}:")
    print(auto.results[i]["best_name"], auto.results[i]["oof_bal_acc"])

Region 3:
CatBoost 0.8133182844243793
Region 4:
CatBoost 0.7044534412955465
Region 5:
CatBoost 0.8598356694055099
Region 6:
CatBoost 0.827186512118019
Region 7:
CatBoost 0.8862660944206009
Region 8:
CatBoost 1.0
Region 9:
CatBoost 0.8835376532399299
Region 10:
CatBoost 0.8486279138388906
Region 11:
CatBoost 0.8361086765994742
Region 12:
CatBoost 0.8648519579751671
Region 13:
CatBoost 0.7871621621621622
Region 14:
CatBoost 0.8659629843840371
Region 15:
CatBoost 0.8107370336669699
Region 16:
CatBoost 0.7854984894259819
Region 17:
CatBoost 0.7821229050279329
Region 18:
CatBoost 0.8219037871033776
Region 19:
CatBoost 1.0
Region 20:
CatBoost 0.7611607142857143
Region 21:
CatBoost 0.7911663807890223
Region 22:
CatBoost 0.675


# Get predictions

In [24]:
all_cls = all_preds.set_index('SamplingOperations_code')
yhat_te = all_cls['yhat_te'][all_cls['yhat_te'].notna()]
yhat_te = yhat_te.to_frame()
yhat_te.to_csv('Status_2_region.csv')
yhat_te

,yhat_te
SamplingOperations_code,
S05169000_20130911,High
S05169000_20200819,High
S05172000_20210813,Good
S05172050_20210824,Good
S05172350_20120827,High
...,...
S03128500_20120723,Moderate
S03128640_20130718,Moderate
S03128640_20181011,Moderate


In [25]:
# For all regions 1, 2, ..., 22
regiondfs = {}
for region in range(1, 3):
    print(f"Region: {region}")
    regiondf = df[df['HERlvl1Code'] == region]

    cleanregion = cleandf.IWANTMYXCLEAN(regiondf, thresh_high_missing=.90)
    regiondfs[region] = cleanregion

auto = MultiRegionBoostedClassifier(regiondfs, random_state=42, verbose=True, models=["XGBoost"])
auto.process_missing_class(scoring="accuracy", n_iter_per_model=3, max_splits=5, calibrate=True)

Region: 1
Dropped exact duplicate columns: ['Achac01', 'Achal01', 'Achca01', 'Achch01', 'Achcl01', 'Achco01', 'Achco03', 'Achcy01', 'Achde01', 'Achde02', 'Achdi01', 'Achdi02', 'Achel01', 'Achen01', 'Achex03', 'Achfl01', 'Achfr01', 'Achgr01', 'Achgr02', 'Achha01', 'Achhe01', 'Achhi01', 'Achho01', 'Achho02', 'Achim01', 'Achim02', 'Achim03', 'Achin01', 'Achin02', 'Achjo01', 'Achko01', 'Achkr01', 'Achkr03', 'Achkr04', 'Achku01', 'Achla01', 'Achla05', 'Achle01', 'Achle02', 'Achli02', 'Achli04', 'Achlo01', 'Achlu01', 'Achlu02', 'Achmo01', 'Achmo02', 'Achna01', 'Achna02', 'Achni01', 'Achno01', 'Achpa01', 'Achpe01', 'Achpe02', 'Achpl01', 'Achpo01', 'Achpr01', 'Achpr02', 'Achps01', 'Achps02', 'Achps03', 'Achpu01', 'Achro01', 'Achro03', 'Achru02', 'Achsc01', 'Achse01', 'Achse02', 'Achsi01', 'Achst01', 'Achst02', 'Achst04', 'Achsu02', 'Achsu04', 'Achtu01', 'Achzi01', 'Actde01', 'Actno01', 'Actro01', 'Actse01', 'Actsp01', 'Actsu01', 'Actvu01', 'Adlaq01', 'Adlbr01', 'Adlmu01', 'Adlmu02', 'Adlpa01',

In [26]:
manifest = auto.save_all(outdir="models", compress=3, make_figs=True)
manifest

,region,best_name,calibrated_path,encoder_path,meta_path,viz_dir,oof_acc,oof_f1_macro
0,1,XGBoost,models\best_xgboost_1.joblib,models\best_xgboost_1_label_encoder.joblib,models\best_xgboost_1_meta.json,models\best_xgboost_1_viz,0.99543,0.729089
1,2,XGBoost,models\best_xgboost_2.joblib,models\best_xgboost_2_label_encoder.joblib,models\best_xgboost_2_meta.json,models\best_xgboost_2_viz,1.00000,1.000000


In [27]:
# all regions
reports = auto.save_all_reports(outdir="models")
reports

,region,best_name,report_dir,oof_acc,oof_f1_macro
0,1,XGBoost,models\best_xgboost_1_viz,0.99543,0.729089
1,2,XGBoost,models\best_xgboost_2_viz,1.00000,1.000000


In [28]:
all_preds = auto.all_inferred()
all_preds.to_csv("all_regions_inferred.csv", index=False)

In [29]:
all_cls = all_preds.set_index('SamplingOperations_code')
yhat_te = all_cls['yhat_te'][all_cls['yhat_te'].notna()]
yhat_te = yhat_te.to_frame()
yhat_te.to_csv('Status_catboost.csv')
yhat_te

,yhat_te
SamplingOperations_code,
S05169000_20130911,High
S05169000_20200819,High
S05172000_20210813,Good
S05172050_20210824,Good
S05172350_20120827,High
...,...
S06700075_20120611,High
S06700094_20210830,High
S06700590_20210830,Moderate
